Step 1: Install and Import MONAI

In [ ]:
!pip install monai nibabel

import os
import torch
import monai
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, CropForegroundd,
    Spacingd, Resized, NormalizeIntensityd, RandSpatialCropSamplesd,
    RandFlipd, RandShiftIntensityd, MapTransform
)
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.data import Dataset, DataLoader, decollate_batch
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

Step 2: Label conversion class (Same as original Notebook)

In [ ]:
class ConvertToMultiChannelBasedOnBratsClassesd(MapTransform):
    """Chuyển đổi nhãn 1, 2, 4 thành 3 kênh: TC, WT, ET"""
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = []
            # Lõi khối u (Tumor Core - TC): nhãn 1 hoặc 4
            result.append(torch.logical_or(d[key] == 1, d[key] == 4))
            # Toàn bộ u (Whole Tumor - WT): nhãn 1, 2, hoặc 4
            result.append(torch.logical_or(torch.logical_or(d[key] == 1, d[key] == 4), d[key] == 2))
            # U ngấm thuốc (Enhancing Tumor - ET): nhãn 4
            result.append(d[key] == 4)
            d[key] = torch.cat(result, axis=0).float()
        return d

Step 3: Create 3D Data Pipeline

In [ ]:
import os

def load_datalist(root_dir: str):
    """
    Quét qua các thư mục bệnh nhân, kiểm tra file tồn tại
    và gom nhóm 4 ảnh MRI + 1 ảnh Label.
    """
    model_scans = ["flair", "t1", "t1ce", "t2"]
    datalist = []

    for data in os.listdir(root_dir):
        data_dir = os.path.join(root_dir, data)
        if not os.path.isdir(data_dir):
            continue

        # Chú ý: Đuôi file của bộ này là .nii
        image_paths = [os.path.join(data_dir, f"{data}_{scan}.nii") for scan in model_scans]
        label_path = os.path.join(data_dir, f"{data}_seg.nii")

        # RẤT QUAN TRỌNG: Chỉ thêm vào list nếu TẤT CẢ 5 file đều tồn tại
        if all(os.path.exists(p) for p in image_paths + [label_path]):
            datalist.append({
                "image": image_paths,
                "label": label_path
            })

    return datalist

# 1. Tạo danh sách dữ liệu an toàn
DATA_DIR = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
data_dicts = load_datalist(DATA_DIR)
print(f"Tổng số case MRI hợp lệ thu thập được: {len(data_dicts)}")

# 2. Chia Train / Val (Lấy 80% Train, 20% Val)
train_size = int(len(data_dicts) * 0.8)
train_files, val_files = data_dicts[:train_size], data_dicts[train_size:]
print(f"Train size: {len(train_files)} | Val size: {len(val_files)}")

# 3. ĐỊNH NGHĨA PHÉP BIẾN ĐỔI (TRANSFORMS)
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["label"]),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    # Cắt các khối nhỏ 96x96x96 để nhét vừa VRAM GPU
    RandSpatialCropSamplesd(keys=["image", "label"], roi_size=(96, 96, 96), random_size=False, num_samples=2),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["label"]),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

# 4. DataLoader
train_ds = Dataset(data=train_files, transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)

val_ds = Dataset(data=val_files, transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

Step 4: Initialize SwinUNETR and Train Loop

In [ ]:
# KHỞI TẠO MÔ HÌNH SWIN-UNETR CỦA MONAI
model = SwinUNETR(
    in_channels=4, # 4 kênh MRI
    out_channels=3, # 3 kênh nhãn (TC, WT, ET)
    feature_size=24,
    use_checkpoint=True,
    spatial_dims=3,
).to(device)

loss_function = DiceCELoss(to_onehot_y=False, sigmoid=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
dice_metric = DiceMetric(include_background=False, reduction="mean")

MAX_EPOCHS = 10
best_metric = -1

for epoch in range(MAX_EPOCHS):
    # ==== TRAINING ====
    model.train()
    epoch_loss = 0
    step = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]")

    for batch_data in pbar:
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})

    print(f"Epoch {epoch+1} Average Loss: {epoch_loss/step:.4f}")

    # ==== VALIDATION ====
    model.eval()
    with torch.no_grad():
        pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Val]")
        for val_data in pbar_val:
            val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)

            # Sliding Window Inference: Dùng để dự đoán ảnh to từ các patch nhỏ
            val_outputs = sliding_window_inference(val_inputs, (96, 96, 96), 4, model)
            val_outputs = (val_outputs.sigmoid() > 0.5).float() # Threshold 0.5

            dice_metric(y_pred=val_outputs, y=val_labels)

        metric = dice_metric.aggregate().item()
        dice_metric.reset()

        print(f"Validation Mean Dice: {metric:.4f}")

        if metric > best_metric:
            best_metric = metric
            torch.save(model.state_dict(), "best_swinunetr_brats.pth")
            print(f"SAVE THE BEST NEW MODEL! Dice: {best_metric:.4f}")